**AIAC Fresh Graduate Assessment | Task Code: AIAC-GA-2025-01**

Author: Abdulaziz Mohammed AL-Shuieli | Date: 2025 | University of Nizwa

**How to run:** Runtime → Change runtime type → T4 GPU → then Run All

# VisionCare AI — Automated Eye Disease Detection System (Fixed Version)

السبب الرئيسي لفشل النموذج السابق كان استخدام `rescale=1./255` مع EfficientNet.
EfficientNet له طبقة Normalization مدمجة تتوقع بكسلات في النطاق [0, 255].
هذه النسخة تستخدم `preprocess_input` الصحيح + تقسيم البيانات قبل الـ oversampling.

## Step 1 — Mount Google Drive

In [ ]:
# استيراد البيانات من الدرايف
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully.')

## Step 2 — Install Libraries & Imports

In [ ]:
# تثبيت الحزم الناقصة
import subprocess
subprocess.run(['pip', 'install', '-q', 'mlflow', 'scikit-learn',
                'seaborn', 'matplotlib', 'tensorflow', 'pandas', 'numpy', 'opencv-python'])

# تثبيت المكتبات (لو دايما تثبت)
import os
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image as PILImage

# Scikit-learn
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, regularizers
from tensorflow.keras.applications import EfficientNetB3
# preprocess_input مهم جدا — يحل المشكلة الأساسية في النسخة السابقة
from tensorflow.keras.applications.efficientnet import preprocess_input as efn_preprocess
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, img_to_array, load_img
)

# MLflow لتتبع التجارب (ميزة إضافية)
import mlflow
import mlflow.keras

# تكرار
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
warnings.filterwarnings('ignore')

# mixed precision عشان السرعة على T4
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision enabled.')
except Exception as e:
    print('Mixed precision unavailable:', e)

print('TensorFlow version :', tf.__version__)
print('GPU available      :', tf.config.list_physical_devices('GPU'))
print('All imports successful.')

## Step 3 — Project Configuration

In [ ]:
# جميع إعدادات المشروع في مكان واحد - يسهل تغييرها دون الحاجة إلى البحث في الكود
DATA_PATH        = Path('/content/drive/MyDrive/eys_data_project/data-2/Augmented_Dataset')
SPLIT_PATH       = Path('/content/eye_split')             # تقسيم البيانات الأصلية train/val
BALANCED_TRAIN   = Path('/content/eye_train_balanced')    # هنا نسوي oversampling على التدريب فقط
OUTPUT_DIR       = Path('/content/visioncare_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# EfficientNetB3 يتوقع 300x300
IMG_SIZE         = 300
BATCH_SIZE       = 24            # B3 أثقل من B0 — batch أصغر
TARGET_PER_CLASS = 2500          # نوازن كل فئة في التدريب لهذا الرقم
VAL_SPLIT        = 0.20

EPOCHS_P1        = 15            # phase 1: head only
EPOCHS_P2        = 40            # phase 2: fine-tune
LR_P1            = 1e-3
LR_P2            = 1e-4          # ليس 1e-5 — كان قليل جدا في النسخة السابقة
UNFREEZE_LAYERS  = 80            # نفك أخر N طبقات

print('Dataset path :', DATA_PATH)
print('Output dir   :', OUTPUT_DIR)
print('Image size   :', IMG_SIZE)
print('Batch size   :', BATCH_SIZE)

## Step 4 — Verify Dataset Access

In [ ]:
# تشييك ع السريع ع البيانات و الكلاسس
if not DATA_PATH.exists():
    raise FileNotFoundError('Dataset not found. Check Drive is mounted and path is correct.')

classes = sorted([d for d in os.listdir(DATA_PATH)
                  if (DATA_PATH / d).is_dir() and not d.startswith('.')])
print('Classes found ({}) :'.format(len(classes)))
for c in classes:
    n = len(list((DATA_PATH / c).glob('*.*')))
    print('  {:<45} {:>5} images'.format(c, n))

Assessment requires at least 4 visualisations. We produce 5.

## Step 5 — Exploratory Data Analysis (EDA)

In [ ]:
# --- جمع الإحصاءات باستخدام مكتبة Pandas لتحليل جدولي دقيق ---
records = []
for cls in classes:
    imgs = list((DATA_PATH / cls).glob('*.*'))
    records.append({'Class': cls, 'Count': len(imgs)})

df_dist = pd.DataFrame(records).sort_values('Count', ascending=False).reset_index(drop=True)
df_dist['Percentage'] = (df_dist['Count'] / df_dist['Count'].sum() * 100).round(2)
print(df_dist.to_string(index=False))
print('\nTotal images:', df_dist['Count'].sum())

# ── VIS 1: مخطط توزيع الفئات
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#EF4444' if c == 'Pterygium' else '#0D9488' for c in df_dist['Class']]
bars = ax.bar(df_dist['Class'], df_dist['Count'], color=colors, edgecolor='black', linewidth=0.6)
for bar, val in zip(bars, df_dist['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(val), ha='center', fontsize=8, fontweight='bold')
ax.set_title('VisionCare AI — Class Distribution (Red = Underrepresented)', fontsize=13, fontweight='bold')
ax.set_ylabel('Image Count')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── VIS 2: شبكة الصور النموذجية
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('VisionCare AI — Sample Images Per Class', fontsize=14, fontweight='bold')
for i, cls in enumerate(classes):
    imgs = list((DATA_PATH / cls).glob('*.jpg')) + list((DATA_PATH / cls).glob('*.png'))
    if imgs:
        img = PILImage.open(random.choice(imgs)).resize((224, 224))
        axes.flat[i].imshow(img)
        axes.flat[i].set_title(cls, fontsize=8, fontweight='bold')
        axes.flat[i].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_2_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

# ── VIS 3: مخطط Pie
fig, ax = plt.subplots(figsize=(10, 8))
ax.pie(df_dist['Count'], labels=[c.replace(' ', '\n') for c in df_dist['Class']],
       autopct='%1.1f%%', startangle=140,
       colors=plt.cm.Set3(np.linspace(0, 1, len(df_dist))),
       textprops={'fontsize': 8})
ax.set_title('Class Proportions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_3_pie_chart.png', dpi=150, bbox_inches='tight')
plt.show()

# ── VIS 4: توزيع Pixel Intensity
fig, ax = plt.subplots(figsize=(13, 5))
for cls in classes:
    imgs = list((DATA_PATH / cls).glob('*.jpg'))
    if not imgs:
        continue
    sample = random.sample(imgs, min(40, len(imgs)))
    px = []
    for p in sample:
        px.extend(np.array(PILImage.open(p).resize((64, 64)).convert('L')).flatten().tolist())
    ax.hist(px, bins=50, alpha=0.35, label=cls, density=True)
ax.set_title('Pixel Intensity Distribution Per Class', fontsize=13, fontweight='bold')
ax.set_xlabel('Pixel Value (0-255)')
ax.set_ylabel('Density')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_4_pixel_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

# ── VIS 5: خريطة حرارية للبيانات غير المتزنة
df_ratio = df_dist.copy()
df_ratio['Imbalance Ratio'] = (df_ratio['Count'].max() / df_ratio['Count']).round(1)
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=df_ratio, x='Class', y='Imbalance Ratio',
            palette=['#EF4444' if r > 10 else '#F59E0B' if r > 3 else '#0D9488'
                     for r in df_ratio['Imbalance Ratio']], ax=ax)
ax.axhline(y=1, color='black', linestyle='--', linewidth=1.2, label='Balanced (ratio=1)')
ax.set_title('Class Imbalance Ratio (Higher = More Imbalanced)', fontsize=12, fontweight='bold')
ax.set_ylabel('Max Count / Class Count')
plt.xticks(rotation=35, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_5_imbalance_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA complete. 5 visualisations saved.')

Pterygium has only 102 images. لكن نسوي split أولا قبل ما نوازن البيانات
عشان لا تطلع نسخ معززة من نفس الصورة في التدريب و الـ validation (data leakage).

## Step 6 — Stratified Train/Val Split (قبل الـ Oversampling)

In [ ]:
# نسوي تقسيم على الصور الأصلية أولا — كل فئة بنسبة 80/20
for p in [SPLIT_PATH, BALANCED_TRAIN]:
    if p.exists():
        shutil.rmtree(p)

(SPLIT_PATH / 'train').mkdir(parents=True, exist_ok=True)
(SPLIT_PATH / 'val').mkdir(parents=True, exist_ok=True)

for cls in classes:
    src_imgs = sorted(list((DATA_PATH / cls).glob('*.jpg')) +
                      list((DATA_PATH / cls).glob('*.jpeg')) +
                      list((DATA_PATH / cls).glob('*.png')))
    if len(src_imgs) < 5:
        print('WARNING: {} has only {} images'.format(cls, len(src_imgs)))
        continue

    # تقسيم مستقر (stratified بطبيعته لأن كل فئة مفصولة عن الباقي)
    train_files, val_files = train_test_split(
        src_imgs, test_size=VAL_SPLIT, random_state=SEED, shuffle=True
    )

    (SPLIT_PATH / 'train' / cls).mkdir(parents=True, exist_ok=True)
    (SPLIT_PATH / 'val'   / cls).mkdir(parents=True, exist_ok=True)

    for f in train_files:
        shutil.copy(f, SPLIT_PATH / 'train' / cls / f.name)
    for f in val_files:
        shutil.copy(f, SPLIT_PATH / 'val' / cls / f.name)

    print('{:<45} train={:>4}   val={:>4}'.format(cls, len(train_files), len(val_files)))

print('\nSplit done.')

## Step 7 — Balance Dataset (Oversample التدريب فقط)

نعزز فقط مجموعة التدريب. الـ validation يبقى صور حقيقية أصلية بدون تعديل.

In [ ]:
aug_gen = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.25,
    brightness_range=[0.7, 1.3],
    shear_range=0.15,
    fill_mode='nearest'
    # rescale=None — نحفظ البكسل الخام عشان EfficientNet يحتاج [0, 255]
)

train_src = SPLIT_PATH / 'train'

for cls_dir in sorted(train_src.iterdir()):
    if not cls_dir.is_dir() or cls_dir.name.startswith('.'):
        continue
    out_cls = BALANCED_TRAIN / cls_dir.name
    out_cls.mkdir(parents=True, exist_ok=True)

    images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpeg'))
    print('\n{:<45} original: {}'.format(cls_dir.name, len(images)))

    # ننسخ الأصليات أولا
    for p in images:
        dst = out_cls / p.name
        if not dst.exists():
            shutil.copy(p, dst)

    # ننشئ صور معززة لين نوصل الهدف عشان نسوي ميزانية للداتا
    current = len(list(out_cls.iterdir()))
    needed = TARGET_PER_CLASS - current
    if needed > 0:
        print('  -> generating {} more...'.format(needed))
        generated = 0
        while generated < needed:
            src = random.choice(images)
            img = load_img(src, target_size=(IMG_SIZE, IMG_SIZE))
            x = np.expand_dims(img_to_array(img), axis=0)
            for batch in aug_gen.flow(x, batch_size=1):
                PILImage.fromarray(batch[0].astype('uint8')).save(
                    out_cls / 'aug_{:06d}.jpg'.format(generated))
                generated += 1
                if generated >= needed:
                    break
    print('  -> final: {}'.format(len(list(out_cls.iterdir()))))

print('\nBalancing done!')

## Step 8 — Data Generators

هنا الإصلاح الأهم: نستخدم `efn_preprocess` بدل `rescale=1./255`
لأن EfficientNet عنده Normalization layer مدمجة تتوقع بكسلات [0, 255].

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=efn_preprocess,   # <-- الإصلاح الأهم
    rotation_range=15,
    horizontal_flip=True,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=efn_preprocess    # <-- نفس preprocessing بدون augmentation
)

train_gen = train_datagen.flow_from_directory(
    BALANCED_TRAIN, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED
)
val_gen = val_datagen.flow_from_directory(
    SPLIT_PATH / 'val', target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)

NUM_CLASSES  = len(train_gen.class_indices)
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
class_names  = [idx_to_class[i] for i in range(NUM_CLASSES)]

# ملاحظة: ما نحتاج class_weight لأن البيانات متوازنة فعلا بالـ oversampling
print('Training samples  :', train_gen.samples)
print('Validation samples:', val_gen.samples)
print('Num classes       :', NUM_CLASSES)
print('Class map         :', train_gen.class_indices)

Required by assessment. Establishes the performance floor.

## Step 9 — Baseline Model (Dummy Classifier)

In [ ]:
# Flatten عشان نستخدمها في مكتبة sklearn
# DummyClassifier ما يحتاج features حقيقية — يستخدم توزيع الفئات فقط
y_val_true     = val_gen.classes
y_train_labels = train_gen.classes

dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(np.zeros((len(y_train_labels), 1)), y_train_labels)
y_dummy = dummy.predict(np.zeros((len(y_val_true), 1)))

baseline_acc = accuracy_score(y_val_true, y_dummy)
baseline_f1  = f1_score(y_val_true, y_dummy, average='macro', zero_division=0)
print('Baseline Accuracy :', round(baseline_acc, 4))
print('Baseline Macro F1 :', round(baseline_f1, 4))
print('(This is the floor our models must beat.)')

Assessment requires minimum k=5. We run this on a lightweight feature extraction pass
(frozen EfficientNetB3 features + logistic head) to satisfy the requirement efficiently
without re-training the full model 5 times.

## Step 10 — K-Fold Cross-Validation (k=5)

In [ ]:
print('Extracting features for cross-validation (this may take a few minutes)...')

# نستخدم EfficientNetB3 مجمدة كـ feature extractor
feature_extractor = EfficientNetB3(
    include_top=False, weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling='avg'
)

# نجمع الميزات بو الصور — نفس الـ preprocessing الصحيح
full_gen = ImageDataGenerator(
    preprocessing_function=efn_preprocess
).flow_from_directory(
    BALANCED_TRAIN, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=64, class_mode='sparse', shuffle=False
)

X_feats, y_feats = [], []
for i, (batch_x, batch_y) in enumerate(full_gen):
    feats = feature_extractor.predict(batch_x, verbose=0)
    X_feats.append(feats)
    y_feats.append(batch_y)
    if (i + 1) % 20 == 0:
        print('  batch {}/{}'.format(i+1, len(full_gen)))
    if (i + 1) >= len(full_gen):
        break

X_feats = np.vstack(X_feats)
y_feats = np.concatenate(y_feats).astype(int)
print('Feature matrix shape:', X_feats.shape)

# 5-fold stratified cross-validation with logistic regression head
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1))
])
cv_scores = cross_val_score(pipe, X_feats, y_feats, cv=skf,
                            scoring='accuracy', n_jobs=-1, verbose=1)

print('\n5-Fold CV Accuracy Scores:', [round(s, 4) for s in cv_scores])
print('Mean Accuracy : {:.4f}'.format(cv_scores.mean()))
print('Std Deviation : {:.4f}'.format(cv_scores.std()))

# نحفظ الجدول للتقرير
df_cv = pd.DataFrame({'Fold': list(range(1, 6)), 'Accuracy': cv_scores})
df_cv.loc[len(df_cv)] = ['Mean', cv_scores.mean()]
print(df_cv.to_string(index=False))
df_cv.to_csv(OUTPUT_DIR / 'kfold_results.csv', index=False)

Tracks all hyperparameters, metrics, and model artifacts automatically.

## Step 11 — MLflow Experiment Tracking (Bonus)

In [ ]:
mlflow.set_experiment('VisionCare_AI_Fixed')
print('MLflow experiment set: VisionCare_AI_Fixed')
print('Tracking URI:', mlflow.get_tracking_uri())

## Step 12 — Build EfficientNetB3 Model

In [ ]:
base_model = EfficientNetB3(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False    # freeze during phase 1

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(512, activation='relu',
                       kernel_regularizer=regularizers.l2(1e-4))(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.3)(x)
# الـ output layer لازم float32 مع mixed precision عشان الاستقرار العددي
outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

model = tf.keras.Model(inputs, outputs, name='VisionCare_EfficientNetB3')
model.summary()

Base frozen. Only the classifier head trains. LR = 0.001

## Step 13 — Training Phase 1: Head Only

In [ ]:
model.compile(
    optimizer=optimizers.Adam(LR_P1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb1 = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=2, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(str(OUTPUT_DIR / 'phase1_best.keras'),
                              monitor='val_accuracy', save_best_only=True, verbose=1)
]

# log to MLflow
with mlflow.start_run(run_name='EfficientNetB3_Phase1'):
    mlflow.log_params({
        'phase': 1, 'learning_rate': LR_P1,
        'epochs': EPOCHS_P1, 'batch_size': BATCH_SIZE,
        'base_frozen': True, 'img_size': IMG_SIZE,
        'architecture': 'EfficientNetB3'
    })
    history1 = model.fit(
        train_gen, epochs=EPOCHS_P1,
        validation_data=val_gen,
        callbacks=cb1, verbose=1
    )
    best_p1 = max(history1.history['val_accuracy'])
    mlflow.log_metric('best_val_accuracy_phase1', best_p1)

print('Phase 1 best val_accuracy: {:.2%}'.format(best_p1))

Unfreeze top 80 layers. LR = 0.0001 (مو 0.00001 — كان قليل جدا).

## Step 14 — Training Phase 2: Fine-tune Top Layers

In [ ]:
# نفك آخر N طبقات
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

# نحافظ على BatchNorm مجمدة — مهم للاستقرار
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print('Trainable layers after unfreezing:',
      sum(1 for l in model.layers if l.trainable))

model.compile(
    optimizer=optimizers.Adam(LR_P2),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb2 = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=8,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                patience=3, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint(str(OUTPUT_DIR / 'visioncare_efficientnetb3.keras'),
                              monitor='val_accuracy', save_best_only=True, verbose=1)
]

with mlflow.start_run(run_name='EfficientNetB3_Phase2'):
    mlflow.log_params({
        'phase': 2, 'learning_rate': LR_P2,
        'epochs': EPOCHS_P2, 'unfrozen_layers': UNFREEZE_LAYERS
    })
    history2 = model.fit(
        train_gen, epochs=EPOCHS_P2,
        validation_data=val_gen,
        callbacks=cb2, verbose=1
    )
    best_p2 = max(history2.history['val_accuracy'])
    mlflow.log_metric('best_val_accuracy_phase2', best_p2)
    mlflow.keras.log_model(model, 'efficientnetb3_model')

print('Phase 2 best val_accuracy: {:.2%}'.format(best_p2))

## Step 15 — Full Evaluation & Metrics

In [ ]:
val_gen.reset()
y_true = val_gen.classes
y_prob = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

# AUC-ROC (one-vs-rest)
y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
try:
    auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
except Exception:
    auc = float('nan')

print('\n--- EfficientNetB3 Results ---')
print('Accuracy  :', round(acc, 4))
print('Precision :', round(prec, 4))
print('Recall    :', round(rec, 4))
print('F1-Score  :', round(f1, 4))
print('AUC-ROC   :', round(auc, 4))

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix — EfficientNetB3', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=35, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_efficientnetb3.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 16 — Training History Plots

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

all_acc   = history1.history['accuracy']     + history2.history['accuracy']
all_val   = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss  = history1.history['loss']         + history2.history['loss']
all_vloss = history1.history['val_loss']     + history2.history['val_loss']
split     = len(history1.history['accuracy'])

ax1.plot(all_acc, label='Train Accuracy', linewidth=2)
ax1.plot(all_val, label='Val Accuracy', linewidth=2, linestyle='--')
ax1.axvline(x=split, color='red', linestyle=':', label='Fine-tune starts here')
ax1.set_title('Model Accuracy Over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(all_loss,  label='Train Loss', linewidth=2)
ax2.plot(all_vloss, label='Val Loss',   linewidth=2, linestyle='--')
ax2.axvline(x=split, color='red', linestyle=':', label='Fine-tune starts here')
ax2.set_title('Model Loss Over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('VisionCare AI — EfficientNetB3 Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 17 — Custom CNN (للمقارنة الحقيقية بدل أرقام مكتوبة يدويا)

In [ ]:
# Custom CNN يشتغل عادي مع rescale=1./255 لأنه ما عنده Normalization مدمجة
custom_train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    zoom_range=0.1
).flow_from_directory(
    BALANCED_TRAIN, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED
)

custom_val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    SPLIT_PATH / 'val', target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

custom_cnn = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(256, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')
], name='Custom_CNN')

custom_cnn.compile(
    optimizer=optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb_cnn = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=3, min_lr=1e-6, verbose=1)
]

print('Training Custom CNN...')
hist_cnn = custom_cnn.fit(
    custom_train_gen, epochs=15,
    validation_data=custom_val_gen,
    callbacks=cb_cnn, verbose=1
)

# نقيم الـ Custom CNN
custom_val_gen.reset()
y_true_cnn = custom_val_gen.classes
y_prob_cnn = custom_cnn.predict(custom_val_gen, verbose=1)
y_pred_cnn = np.argmax(y_prob_cnn, axis=1)

cnn_acc  = accuracy_score(y_true_cnn, y_pred_cnn)
cnn_prec = precision_score(y_true_cnn, y_pred_cnn, average='macro', zero_division=0)
cnn_rec  = recall_score(y_true_cnn, y_pred_cnn, average='macro', zero_division=0)
cnn_f1   = f1_score(y_true_cnn, y_pred_cnn, average='macro', zero_division=0)
try:
    y_bin_cnn = label_binarize(y_true_cnn, classes=list(range(NUM_CLASSES)))
    cnn_auc   = roc_auc_score(y_bin_cnn, y_prob_cnn, average='macro', multi_class='ovr')
except Exception:
    cnn_auc = float('nan')

print('\nCustom CNN — Acc: {:.4f}  F1: {:.4f}  AUC: {:.4f}'.format(cnn_acc, cnn_f1, cnn_auc))

## Step 18 — Model Comparison Table

In [ ]:
# ننشئ جدول مقارنة نظيف باستخدام مكتبة Pandas — كل الأرقام حقيقية من الـ training
results = [
    {'Model': 'Baseline (Dummy)', 'Accuracy': round(baseline_acc, 4),
     'Precision': 0.0, 'Recall': 0.0,
     'F1-Score': round(baseline_f1, 4), 'AUC-ROC': 'N/A'},
    {'Model': 'Custom CNN', 'Accuracy': round(cnn_acc, 4),
     'Precision': round(cnn_prec, 4), 'Recall': round(cnn_rec, 4),
     'F1-Score': round(cnn_f1, 4), 'AUC-ROC': round(cnn_auc, 4)},
    {'Model': 'EfficientNetB3', 'Accuracy': round(acc, 4),
     'Precision': round(prec, 4), 'Recall': round(rec, 4),
     'F1-Score': round(f1, 4), 'AUC-ROC': round(auc, 4)},
]

df_results = pd.DataFrame(results)
print('\n===== Final Model Comparison =====')
print(df_results.to_string(index=False))

# نحفظ للتقرير
df_results.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
print('\nSaved to model_comparison.csv')

# Bar chart مرئي
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(df_results))
w = 0.2
ax.bar(x - 1.5*w, df_results['Accuracy'],                w, label='Accuracy')
ax.bar(x - 0.5*w, df_results['Precision'].astype(float), w, label='Precision')
ax.bar(x + 0.5*w, df_results['Recall'].astype(float),    w, label='Recall')
ax.bar(x + 1.5*w, df_results['F1-Score'],                w, label='F1-Score')
ax.set_xticks(x)
ax.set_xticklabels(df_results['Model'])
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison (Validation Set)', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

Highlights which regions of the retinal image influenced the model's prediction.

## Step 19 — Grad-CAM Explainability (XAI — Bonus +5 pts)

In [ ]:
import cv2

def find_last_conv_layer(full_model):
    # ندور آخر طبقة conv داخل الـ base model
    base = None
    for layer in full_model.layers:
        if isinstance(layer, tf.keras.Model):
            base = layer
            break
    if base is None:
        base = full_model
    for layer in reversed(base.layers):
        if 'conv' in layer.name.lower() or isinstance(layer, layers.Conv2D):
            return base, layer.name
    return base, base.layers[-1].name


def make_gradcam(img_array, full_model):
    base, last_conv_name = find_last_conv_layer(full_model)

    grad_model = tf.keras.Model(
        inputs=full_model.inputs,
        outputs=[base.get_layer(last_conv_name).output, full_model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array, training=False)
        pred_idx = tf.argmax(preds[0])
        class_ch = preds[:, pred_idx]

    grads   = tape.gradient(class_ch, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_idx), float(preds[0][pred_idx])


fig, axes = plt.subplots(2, 5, figsize=(20, 9))
fig.suptitle('VisionCare AI — Grad-CAM: What the Model Sees', fontsize=14, fontweight='bold')
axes = axes.flatten()

# نستخدم صور من الـ validation (أصلية بدون تعزيز)
for i, cls in enumerate(class_names[:10]):
    imgs = list((SPLIT_PATH / 'val' / cls).glob('*.jpg')) +            list((SPLIT_PATH / 'val' / cls).glob('*.png'))
    if not imgs:
        axes[i].axis('off')
        continue

    img_path = random.choice(imgs)
    pil_img  = PILImage.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_raw  = np.array(pil_img).astype(np.float32)
    # نفس الـ preprocessing تبع EfficientNet
    arr      = efn_preprocess(np.expand_dims(img_raw, axis=0).copy())

    try:
        heatmap, pred_idx, conf = make_gradcam(arr, model)
        hm_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
        hm_colored = cv2.applyColorMap(np.uint8(255 * hm_resized), cv2.COLORMAP_JET)
        hm_colored = cv2.cvtColor(hm_colored, cv2.COLOR_BGR2RGB)
        overlay    = cv2.addWeighted(img_raw.astype('uint8'), 0.6, hm_colored, 0.4, 0)

        axes[i].imshow(overlay)
        pred_name = idx_to_class[pred_idx]
        color = 'green' if pred_name == cls else 'red'
        axes[i].set_title('True: {}\nPred: {} ({:.0%})'.format(cls, pred_name, conf),
                          fontsize=7, fontweight='bold', color=color)
    except Exception as e:
        axes[i].imshow(img_raw.astype('uint8'))
        axes[i].set_title('{}\n(gradcam error)'.format(cls), fontsize=8)

    axes[i].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gradcam.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 20 — Final Summary

In [ ]:
# ملخص نهائي لكل المخرجات
summary = {
    'Architecture'              : 'EfficientNetB3',
    'Image Size'                : IMG_SIZE,
    'Batch Size'                : BATCH_SIZE,
    'Num Classes'               : NUM_CLASSES,
    'Training Samples'          : train_gen.samples,
    'Validation Samples'        : val_gen.samples,
    'Baseline Accuracy'         : round(baseline_acc, 4),
    'K-Fold CV Mean Accuracy'   : round(cv_scores.mean(), 4),
    'K-Fold CV Std'             : round(cv_scores.std(), 4),
    'Custom CNN Accuracy'       : round(cnn_acc, 4),
    'Custom CNN F1'             : round(cnn_f1, 4),
    'EfficientNetB3 Accuracy'   : round(acc, 4),
    'EfficientNetB3 Precision'  : round(prec, 4),
    'EfficientNetB3 Recall'     : round(rec, 4),
    'EfficientNetB3 F1'         : round(f1, 4),
    'EfficientNetB3 AUC-ROC'    : round(auc, 4),
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print(summary_df.to_string(index=False))
summary_df.to_csv(OUTPUT_DIR / 'final_summary.csv', index=False)

print('\nAll outputs saved to:', OUTPUT_DIR)
print('Files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print('  -', f.name)